# CP5 - Local LLM layer: root-cause selection (E1, S1) and HyDE (Colab run)

This notebook runs the CP5 evaluation with **free, open language models served locally by Ollama** on a free Colab GPU,
and produces a small zip of result JSON files to hand to the next Claude Code session.

What it does, in order: clone the repository, install Go and the project, rebuild the D1 and D4 evidence on the local
Colab disk (`dataset materialize`) and **stop if it does not match the committed case lists**, embed the D4 chunks with the
fixed retriever's embedder (`bge-small`, cached on Drive), install and start Ollama, pull the models, then for each model run
**E1** (Mode 1: choose from the functions a fix changed vs E0), **S1** (Mode 2: choose from the retriever's top 10 and top 5 vs
its top-1) and **HyDE** (a hypothetical function as a retrieval query), and zip the results.

**Zero cost:** only open models run, on the Colab machine, through `http://localhost:11434/v1`. The project's endpoint rule refuses
any non-local host and every known paid API host. No paid API is called.
**What the zip contains:** result JSON only - parsed selections, validity flags, token counts and latencies. No prompts, no raw
model answers, no hypothetical texts, no repository code, no model files. Raw traces stay in the LLM cache on Drive.

Set `Runtime -> Change runtime type -> T4 GPU` first. Run the cells top to bottom; every step can be re-run safely (finished model
calls come from the cache).

**Running time per step:** not measured yet. The timings table written by the last cells is the real record; please keep it,
it goes into the CP5 report.

**Where files live:** git clones, the vulndb snapshot and the rebuilt evidence are on the local Colab disk (`/content/cache`,
`/content/data`) because git misbehaves on the Google Drive mount. Only the embedding cache, the LLM cache, timings and results go
on Drive. After a disconnect the evidence is rebuilt (about 30 minutes); embeddings and model answers are not.

In [ ]:
# Settings for this run. BRANCH holds the CP5 code. DRY_RUN=True swaps the language model and the embedder for deterministic
# fakes (a quick end-to-end check with no model download and no Ollama). MODELS are the Ollama tags to compare (at most two).
BRANCH = "claude/vulntrace-ai-build-o31zls"
REPO_URL = "https://github.com/AMAN-PRIYANSH/REPO-MIND.git"
DRY_RUN = False
GO_VERSION = "1.24.7"
MODELS = ["qwen2.5-coder-7b", "llama3.1-8b"]      # aliases defined in the project's models_config
CONTEXT_TOKENS = 32768                             # Ollama context length; the largest D1 prompt is about 13k estimated tokens, real tokenizers count more
K_VALUES = [10, 5]                                 # S1: top-k shown to the model (10 is the headline, 5 the ablation)

In [ ]:
# A tiny timer so the real running time of every step is recorded (written to Drive and included in the results zip).
import json, os, shutil, subprocess, sys, time
class Timings:
    def __init__(self): self.rows, self.t0, self.label = [], None, None
    def start(self, label): self.label, self.t0 = label, time.time(); print(">>>", label)
    def stop(self):
        sec = round(time.time() - self.t0, 1); self.rows.append({"step": self.label, "seconds": sec}); print(f"<<< {self.label}: {sec} s")
T = Timings()

In [ ]:
# Check whether a GPU is attached. Ollama uses it automatically; without one the 7-8B models are far too slow for the full run.
GPU = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print("GPU available:", GPU)
if GPU:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
DEVICE = "cuda" if GPU else "cpu"
if not GPU and not DRY_RUN:
    print("WARNING: no GPU. Switch the runtime to T4 GPU, or set DRY_RUN = True for a pipeline check only.")

In [ ]:
# Mount Google Drive for the files worth keeping across a disconnect: embeddings, the LLM answer cache, timings and results.
# The git cache and the rebuilt evidence stay on the LOCAL disk (git behaves badly on the Drive mount).
from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/vulntrace_cp5"       # Drive: embeddings, LLM cache, results
os.makedirs(WORK, exist_ok=True)
CACHE = "/content/cache"                            # local disk: git clones + vulndb snapshot
DATA = "/content/data"                              # local disk: rebuilt D1 and D4 evidence
os.makedirs(CACHE, exist_ok=True)
os.environ["VULNTRACE_CACHE"] = CACHE
OUT = f"{WORK}/results/cp5" + ("_dryrun" if DRY_RUN else "")
LLM_CACHE = f"{WORK}/llm_cache" + ("_dryrun" if DRY_RUN else "")
DRY = "--dry-run" if DRY_RUN else ""
print("Drive:", WORK, "| local git cache:", CACHE, "| local evidence:", DATA, "| results:", OUT)

In [ ]:
# Clone (or update) the repository at the CP5 branch. The code that does the work lives in the repository; this notebook only calls it.
REPO = "/content/REPO-MIND"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
%cd /content/REPO-MIND

In [ ]:
# Install Go (Colab does not ship it). The project's small Go helper parses Go files to find function boundaries while the
# evidence is rebuilt. Installed under /content only, nothing system-wide.
if not os.path.exists(f"/content/go{GO_VERSION}/go/bin/go"):
    subprocess.run(["wget", "-q", f"https://go.dev/dl/go{GO_VERSION}.linux-amd64.tar.gz", "-O", "/content/go.tgz"], check=True)
    os.makedirs(f"/content/go{GO_VERSION}", exist_ok=True)
    subprocess.run(["tar", "-C", f"/content/go{GO_VERSION}", "-xzf", "/content/go.tgz"], check=True)
os.environ["PATH"] = f"/content/go{GO_VERSION}/go/bin:" + os.environ["PATH"]
os.environ["GOTOOLCHAIN"] = "local"
os.environ["CGO_ENABLED"] = "0"
print(subprocess.run(["go", "version"], capture_output=True, text=True).stdout)

In [ ]:
# Install this project with the [ml] extra (sentence-transformers for the bge-small embedder; Colab already has torch with CUDA).
!pip install -q -e ".[ml]"
# The notebook's Python session does not see the new install until it is restarted, so point it at the checkout directly.
import sys, importlib
sys.path.insert(0, "/content/REPO-MIND/src")
importlib.invalidate_caches()
!python -m vulntrace doctor

In [ ]:
# Fetch the pinned golang/vulndb snapshot and the Go helper's dependencies (both cached; quick on re-runs).
T.start("fetch vulndb + build helper")
!python -m vulntrace dataset fetch
!python -m vulntrace helper fetch
!python -m vulntrace helper build
T.stop()

In [ ]:
# Rebuild the D1 evidence (the changed functions and hunks of each fix commit) for exactly the 150 committed case IDs, on the local
# disk. It never re-selects cases. A finished case is skipped on a re-run only if its stored result matches the committed case
# list; a failed or mismatching one is rebuilt. One pass is enough: the old first-pass mismatch (about 9 cases) came from reusing a
# cached commit that was only the shallow boundary of an earlier fetch, fixed in CP5b. Expect about 6-10 minutes.
T.start("materialize D1")
!python -u -m vulntrace dataset materialize --set d1 --data-dir "{DATA}" --summary-out "{WORK}/D1_materialize.json"
T.stop()

In [ ]:
# Rebuild the D4 evidence (release ranges and candidate code) for the 120 committed case IDs, on the local disk. Slow and
# network-heavy: sequential git fetches with a pause between them. Expect about 25-30 minutes on a 4-core machine.
T.start("materialize D4")
!python -u -m vulntrace dataset materialize --set d4 --data-dir "{DATA}" --summary-out "{WORK}/D4_materialize.json"
T.stop()

In [ ]:
# GATE: stop here if either rebuilt evidence set does not match its committed case list (more than 3 cases with a different status,
# or no candidate functions at all). If this cell stops the notebook, do NOT continue: re-run the two cells above (a mismatching
# case is rebuilt) and read the git errors they print, or run  !python -m vulntrace dataset diagnose --case GO-2020-0038
import json
from collections import Counter
from vulntrace.eval import evidence_gate
problems = []
for name, sub in (("D1", "fix_evidence"), ("D4", "release_range")):
    case_list = json.load(open(f"docs/results/datasets/{name}_cases.json"))
    statuses, n_candidates = evidence_gate.evidence_statuses(case_list, DATA, sub)
    p = evidence_gate.check_evidence(case_list, statuses, n_candidates)
    print(name, "materialised statuses:", dict(Counter(statuses.values())), "| expected:", case_list.get("status_counts"), "| candidates:", n_candidates)
    problems += [f"{name}: {x}" for x in p]
if problems:
    raise SystemExit("STOP - the evidence is not usable: " + "; ".join(problems))
print("gate passed")

In [ ]:
# Embed every D4 chunk (code+commits variant, the one the fixed retriever uses) with bge-small, caching vectors on Drive by the
# chunk text's hash. Skipped in a dry run (the fake embedder needs no precomputation). The GPU matters here: the two largest D4
# ranges have about 14,000 and 15,000 candidate functions each.
if not DRY_RUN:
    T.start("embed D4 chunks (bge-small)")
    !python -u -m vulntrace index build --set d4 --embedder bge-small --variant "code+commits" --data-dir "{DATA}" --embedding-cache "{WORK}/embeddings" --device {DEVICE}
    T.stop()

In [ ]:
# Install and start Ollama on this machine (listening on 127.0.0.1 only) with a 32k-token context, then wait until it answers.
# Skipped in a dry run. zstd is needed by Ollama's installer. `which ollama` only proves the CLI exists (a partial install has no
# runner and fails every generate call with HTTP 500), so the pull cell below also makes one tiny generate call and, if that fails,
# calls reinstall_ollama() once before the run goes on.
import urllib.request
def ollama_up():
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=3).read(); return True
    except Exception:
        return False
def start_ollama():
    env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", OLLAMA_CONTEXT_LENGTH=str(CONTEXT_TOKENS))
    subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/content/ollama.log", "a"), stderr=subprocess.STDOUT)
    for _ in range(60):
        if ollama_up(): break
        time.sleep(2)
def reinstall_ollama():
    subprocess.run("pkill -f 'ollama serve'", shell=True); time.sleep(2)
    subprocess.run("apt-get -qq install -y zstd > /dev/null && curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
    start_ollama()
if not DRY_RUN:
    T.start("install + start Ollama")
    if shutil.which("ollama") is None:
        !apt-get -qq install -y zstd > /dev/null
        !curl -fsSL https://ollama.com/install.sh | sh
    if not ollama_up():
        start_ollama()
    print("ollama answering:", ollama_up())
    print(subprocess.run(["ollama", "--version"], capture_output=True, text=True).stdout)
    T.stop()

In [ ]:
# Pull the models and print the exact tag and digest the server reports. A model that cannot be pulled is recorded in
# model_problems and skipped - there is no fallback to any other service.
from vulntrace.llm import models_config
model_problems, MODEL_INFO = {}, {}
if not DRY_RUN:
    for alias in MODELS:
        tag = models_config.resolve(alias)["ollama_tag"]
        T.start(f"pull {tag}")
        r = subprocess.run(["ollama", "pull", tag], capture_output=True, text=True)
        T.stop()
        if r.returncode != 0:
            model_problems[alias] = (r.stderr or r.stdout)[-500:]; print("PULL FAILED:", alias, model_problems[alias])
    # The install check: one tiny generate call with the first model that pulled. On failure, reinstall once; if it still fails the
    # notebook stops here (OllamaSetupError) instead of failing call by call later.
    from vulntrace.llm import ollama_setup
    pulled = [a for a in MODELS if a not in model_problems]
    if pulled:
        probe_tag = models_config.resolve(pulled[0])["ollama_tag"]
        ollama_setup.ensure_working(lambda: ollama_setup.probe_generate(probe_tag), reinstall_ollama)
    tags = json.load(urllib.request.urlopen("http://127.0.0.1:11434/api/tags"))["models"]
    MODEL_INFO = {m["name"]: {"digest": m["digest"], "size": m["size"], "details": m.get("details")} for m in tags}
    print(json.dumps(MODEL_INFO, indent=1))
MODELS_TO_RUN = ["fake"] if DRY_RUN else [m for m in MODELS if m not in model_problems]
print("models to run:", MODELS_TO_RUN, "| problems:", model_problems or "none")

In [ ]:
# E1 (Mode 1, D1): for each model, the model chooses from the functions the fix commit changed; E0 (all changed functions) is scored
# in the same file. Results: OUT/select_fix__<model>.json. A model call that fails three times in a row stops the run.
for m in MODELS_TO_RUN:
    T.start(f"E1 {m}")
    !python -u -m vulntrace eval select --mode fix --model {m} {DRY} --data-dir "{DATA}" --llm-cache "{LLM_CACHE}" --out "{OUT}"
    T.stop()

In [ ]:
# S1 (Mode 2, D4): for each model and each k, the model chooses from the top k candidates of the fixed retriever (hybrid:bge-small,
# code+commits); the retriever's top-1 is scored in the same file. Results: OUT/select_release__<model>__k<k>.json.
for m in MODELS_TO_RUN:
    for k in K_VALUES:
        T.start(f"S1 {m} k={k}")
        !python -u -m vulntrace eval select --mode release --k {k} --model {m} {DRY} --data-dir "{DATA}" --embedding-cache "{WORK}/embeddings" --device {DEVICE} --llm-cache "{LLM_CACHE}" --out "{OUT}"
        T.stop()

In [ ]:
# HyDE (Mode 2, D4): for each model, one hypothetical function per advisory is written and used as the retrieval query
# (query+hyde, hyde-only), against the plain query, with the fixed retriever and BM25 as a secondary row.
# Results: OUT/hyde__<model>/ (cases.json, hyde_meta.json, runs/*.json).
for m in MODELS_TO_RUN:
    T.start(f"HyDE {m}")
    !python -u -m vulntrace eval retrieval --hyde query+hyde --hyde hyde-only --model {m} {DRY} --data-dir "{DATA}" --embedding-cache "{WORK}/embeddings" --device {DEVICE} --llm-cache "{LLM_CACHE}" --out "{OUT}"
    T.stop()

In [ ]:
# Render the results document from the files just written, to look at it here. The next Claude Code session renders the committed
# copy the same way from the zip.
!python -m vulntrace results render --doc cp5 --input "{OUT}" --out "{WORK}/CP5_LLM_RESULTS_preview.md"
from IPython.display import Markdown, display
display(Markdown(open(f"{WORK}/CP5_LLM_RESULTS_preview.md").read()[:8000]))

In [ ]:
# Zip only the result JSON (selection files, HyDE runs, timings, model digests, evidence summaries, model problems) and download it.
# No prompts, no raw model answers, no hypothetical texts, no code, no models, no embeddings go into the zip. Refuses to build a zip
# if an experiment produced no result file (an empty run must never be handed over as a result).
import zipfile, glob
json.dump(model_problems, open(f"{OUT}/model_problems.json", "w"), indent=1)
json.dump({"models": MODEL_INFO, "timings": T.rows, "dry_run": DRY_RUN, "gpu": GPU, "context_tokens": CONTEXT_TOKENS},
          open(f"{OUT}/colab_run.json", "w"), indent=1)
expected = [f"{OUT}/select_fix__{models_config.resolve(m)['alias']}.json" for m in MODELS_TO_RUN]
expected += [f"{OUT}/select_release__{models_config.resolve(m)['alias']}__k{k}.json" for m in MODELS_TO_RUN for k in K_VALUES]
expected += [f"{OUT}/hyde__{models_config.resolve(m)['alias']}/hyde_meta.json" for m in MODELS_TO_RUN]
missing = [p for p in expected if not os.path.exists(p)]
if missing:
    raise SystemExit("STOP - these result files are missing, not building a zip: " + ", ".join(missing))
zpath = f"{WORK}/cp5_results{'_dryrun' if DRY_RUN else ''}.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(glob.glob(f"{OUT}/**/*.json", recursive=True)):
        z.write(p, "cp5/" + os.path.relpath(p, OUT))
    for n in ("D1_materialize.json", "D4_materialize.json"):
        if os.path.exists(f"{WORK}/{n}"): z.write(f"{WORK}/{n}", f"cp5/{n.replace('.json', '_colab.json')}")
print(zpath, os.path.getsize(zpath) // 1024, "KiB")
from google.colab import files
files.download(zpath)